In [ ]:
import os
import shutil
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

# 1. 固定随机种子 -------------------------------------------------------------
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# 2. 路径常量 -----------------------------------------------------------------
DATA_DIR   = r'D:\test\data'
TRAIN_DIR  = r'D:\test\dataset\train'
VAL_DIR    = r'D:\test\dataset\val'
TEST_DIR   = r'D:\test\dataset\test'
for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(p, exist_ok=True)

# 3. 快速划分（沿用你原函数，仅增加提示） -------------------------------------
def split_dataset(data_dir=DATA_DIR,
                  train_dir=TRAIN_DIR,
                  val_dir=VAL_DIR,
                  test_dir=TEST_DIR,
                  train_ratio=0.7,
                  val_ratio=0.15):
    classes = [c for c in os.listdir(data_dir)
               if os.path.isdir(os.path.join(data_dir, c))]
    if not classes:
        raise ValueError(f'{data_dir} 下没有类别文件夹！')
    for cls in classes:
        src_cls = os.path.join(data_dir, cls)
        train_cls = os.path.join(train_dir, cls)
        val_cls   = os.path.join(val_dir, cls)
        test_cls  = os.path.join(test_dir, cls)
        for p in [train_cls, val_cls, test_cls]:
            os.makedirs(p, exist_ok=True)

        imgs = [f for f in os.listdir(src_cls)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        random.shuffle(imgs)
        n = len(imgs)
        n_train = int(n * train_ratio)
        n_val   = int(n * (train_ratio + val_ratio))
        for i, img in enumerate(imgs):
            src = os.path.join(src_cls, img)
            dst = (train_cls if i < n_train else
                   val_cls   if i < n_val else
                   test_cls)
            shutil.copy(src, dst)
    print('数据集划分完成！')

split_dataset()

# 4. 图像参数 -----------------------------------------------------------------
IMG_SIZE   = 224
BATCH_SIZE = 32

# 5. 数据增强 & 预处理 ---------------------------------------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest')

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False)

NUM_CLASSES = train_gen.num_classes
print(f'类别数: {NUM_CLASSES}')

# 6. 计算类别权重（缓解不平衡） ----------------------------------------------
cls_idx   = train_gen.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(cls_idx),
    y=cls_idx)
class_weights = dict(enumerate(class_weights))
print('类别权重:', class_weights)

# 7. 构建迁移学习模型 ---------------------------------------------------------
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet')
base_model.trainable = False   # 先冻结

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer=Adam(1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

# 8. 回调 ---------------------------------------------------------------------
cbs = [
    callbacks.ReduceLROnPlateau(monitor='val_loss',
                                factor=0.5,
                                patience=3,
                                min_lr=1e-6,
                                verbose=1),
    callbacks.EarlyStopping(monitor='val_loss',
                            patience=6,
                            restore_best_weights=True,
                            verbose=1)
]

# 9. 训练 ---------------------------------------------------------------------
EPOCHS = 30
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=cbs,
    verbose=1)

# 10. 绘图 --------------------------------------------------------------------
def plot_history(h):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(h.history['accuracy'], label='train acc')
    plt.plot(h.history['val_accuracy'], label='val acc')
    plt.legend(), plt.title('Accuracy')
    plt.subplot(1, 2, 2)
    plt.plot(h.history['loss'], label='train loss')
    plt.plot(h.history['val_loss'], label='val loss')
    plt.legend(), plt.title('Loss')
    plt.savefig('training_history.png')
    plt.show()

plot_history(history)

# 11. 测试评估 ----------------------------------------------------------------
test_loss, test_acc = model.evaluate(test_gen)
print(f'测试集准确率: {test_acc:.4f}')

# 12. 保存 --------------------------------------------------------------------
model.save('chinese_medicine_classifier_mobilenetv2.keras')
pd.DataFrame(history.history).to_csv('training_history.csv', index=False)
print('模型与历史已保存！')